# Phase 1 — Canonical ABSA Readiness Audit

This notebook upgrades the audited finance dataset from a structurally valid CSV into an ABSA-ready canonical dataset.

Goals of this phase:

1. verify whether the `Words` column is reliable enough to be retained as metadata;
2. assess the quality of aspect/entity keys extracted from `decisions_norm_json`;
3. determine whether each aspect can be grounded back to the headline text;
4. generate pseudo-spans (`char_from`, `char_to`) whenever reliable matching is possible;
5. separate clean matches from ambiguous cases via a review queue;
6. export a canonical ABSA-ready file for downstream ATE / ASC preparation.

This phase does NOT perform train/val/test splitting yet.
Splitting will only be done after the canonical representation is finalized.

In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

In [2]:
IN_PATH = Path("outputs_phase0/finance_raw_audited.csv")
OUT_DIR = Path("outputs_phase1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_CSV_PATH = OUT_DIR / "finance_absa_phase1_canonical.csv"
CANONICAL_JSONL_PATH = OUT_DIR / "finance_absa_phase1_canonical.jsonl"
WORDS_REPORT_PATH = OUT_DIR / "words_validation_report.json"
ASPECT_QUALITY_PATH = OUT_DIR / "aspect_quality_report.json"
SPAN_REVIEW_QUEUE_PATH = OUT_DIR / "span_review_queue.csv"
PHASE1_REPORT_PATH = OUT_DIR / "phase1_report.json"
PHASE1_CHECKLIST_PATH = OUT_DIR / "phase1_checklist.csv"

## Step 1. Load the audited Phase 0 dataset

We use the Phase 0 audited CSV as input, not the raw CSV.
This ensures that all records have already passed schema validation and sentiment-label normalization.

In [3]:
df = pd.read_csv(IN_PATH)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (10753, 10)


,raw_id,title_raw,title_norm,words_raw,decisions_raw,decisions_norm_json,num_aspects,label_signature,parse_issue,num_invalid_labels
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet to issue 6.4 crore warrants to promoters,8,"{""SpiceJet"": ""neutral""}","{""SpiceJet"": ""neutral""}",1,neutral,ok,0
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC Q2 net loss at Rs 10.4 crore,8,"{""MMTC"": ""neutral""}","{""MMTC"": ""neutral""}",1,neutral,ok,0
2,3,"Mid-cap funds can deliver more, stay put: Experts","Mid-cap funds can deliver more, stay put: Experts",8,"{""Mid-cap funds"": ""positive""}","{""Mid-cap funds"": ""positive""}",1,positive,ok,0
3,4,Mid caps now turn into market darlings,Mid caps now turn into market darlings,7,"{""Mid caps"": ""positive""}","{""Mid caps"": ""positive""}",1,positive,ok,0
4,5,"Market seeing patience, if not conviction: Pra...","Market seeing patience, if not conviction: Pra...",8,"{""Market"": ""neutral""}","{""Market"": ""neutral""}",1,neutral,ok,0


## Step 2. Reconstruct structured decisions from `decisions_norm_json`

The audited CSV stores normalized decisions as JSON strings.
We convert them back into Python dictionaries for aspect-level inspection.

In [4]:
def parse_norm_json(x):
    if pd.isna(x) or str(x).strip() == "":
        return {}
    return json.loads(x)

df["decisions_dict"] = df["decisions_norm_json"].apply(parse_norm_json)
df["aspect_terms"] = df["decisions_dict"].apply(lambda d: list(d.keys()))
df["aspect_sentiments"] = df["decisions_dict"].apply(lambda d: list(d.values()))

print(df[["raw_id", "title_norm", "aspect_terms", "aspect_sentiments"]].head(3))

   raw_id                                         title_norm     aspect_terms  \
0       1  SpiceJet to issue 6.4 crore warrants to promoters       [SpiceJet]   
1       2                  MMTC Q2 net loss at Rs 10.4 crore           [MMTC]   
2       3  Mid-cap funds can deliver more, stay put: Experts  [Mid-cap funds]   

  aspect_sentiments  
0         [neutral]  
1         [neutral]  
2        [positive]  


## Step 3. Validate the `Words` column

We test whether the provided `Words` column is approximately consistent with simple token counts computed from `title_norm`.

This is treated as metadata validation, not as a supervision source.

In [5]:
def safe_word_count_simple(text):
    if pd.isna(text):
        return 0
    text = str(text).strip()
    if text == "":
        return 0
    return len(text.split())

df["word_count_simple"] = df["title_norm"].apply(safe_word_count_simple)
df["words_raw_num"] = pd.to_numeric(df["words_raw"], errors="coerce")
df["word_count_diff"] = df["words_raw_num"] - df["word_count_simple"]
df["word_count_match"] = df["word_count_diff"] == 0

In [6]:
words_summary = {
    "rows_with_numeric_words": int(df["words_raw_num"].notna().sum()),
    "exact_match_count": int(df["word_count_match"].sum()),
    "exact_match_ratio": float(df["word_count_match"].mean()),
    "mean_abs_diff": float(df["word_count_diff"].abs().mean()),
    "max_abs_diff": float(df["word_count_diff"].abs().max()),
}

pprint(words_summary)

{'exact_match_count': 10665,
 'exact_match_ratio': 0.9918162373291175,
 'max_abs_diff': 3.0,
 'mean_abs_diff': 0.008555751883195387,
 'rows_with_numeric_words': 10753}


In [7]:
df_words_mismatch = df.loc[~df["word_count_match"], [
    "raw_id", "title_raw", "words_raw", "word_count_simple", "word_count_diff"
]].copy()

df_words_mismatch.head(20)

,raw_id,title_raw,words_raw,word_count_simple,word_count_diff
33,34,Billionaire's stake shakes Woolworth's buyout ...,7,8,-1
797,798,Anoop Singh to quit as IMF's Asia Pacific Dir...,10,9,1
920,921,Next few years need stable FDI flows: JP Morgan,10,9,1
1127,1128,Markets will move sideways before next upside ...,11,10,1
1283,1284,SAT asks Financial Technologies to approach S...,12,11,1
1323,1324,Forward Markets Commision allows national com...,13,12,1
1338,1339,Infosys president BG Srinivas to join Hong Ko...,13,12,1
1347,1348,Narendra Kothari takes over as new Chairman a...,13,12,1
1371,1372,Expect Moil cash reserves to match its mcap in...,14,13,1
1421,1422,High Court to hear fresh PIL against National...,16,15,1


In [8]:
with open(WORDS_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(words_summary, f, ensure_ascii=False, indent=2)

print("Saved:", WORDS_REPORT_PATH)

Saved: outputs_phase1\words_validation_report.json


## Step 4. Create an aspect-level audit view

The original audited CSV is headline-level.
For ABSA readiness, we now inspect each (headline, aspect, sentiment) pair explicitly.

In [9]:
aspect_rows = []

for _, row in df.iterrows():
    raw_id = row["raw_id"]
    text = row["title_norm"]
    decisions = row["decisions_dict"]
    for term, sentiment in decisions.items():
        aspect_rows.append({
            "raw_id": raw_id,
            "title_norm": text,
            "aspect_term_raw": term,
            "aspect_term_norm": str(term).strip(),
            "sentiment": sentiment,
            "num_aspects_in_headline": row["num_aspects"],
            "label_signature": row["label_signature"],
        })

df_aspect = pd.DataFrame(aspect_rows)
print("Aspect-level rows:", df_aspect.shape)
df_aspect.head()

Aspect-level rows: (14409, 7)


,raw_id,title_norm,aspect_term_raw,aspect_term_norm,sentiment,num_aspects_in_headline,label_signature
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet,SpiceJet,neutral,1,neutral
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC,MMTC,neutral,1,neutral
2,3,"Mid-cap funds can deliver more, stay put: Experts",Mid-cap funds,Mid-cap funds,positive,1,positive
3,4,Mid caps now turn into market darlings,Mid caps,Mid caps,positive,1,positive
4,5,"Market seeing patience, if not conviction: Pra...",Market,Market,neutral,1,neutral


## Step 5. Audit aspect key quality

This step checks whether aspect keys appear clean enough for reliable grounding back to the headline text.

We inspect:
- empty or whitespace-only keys,
- duplicate aspect keys within one headline,
- suspiciously short keys,
- aspect keys containing unusual symbols,
- potential normalization risks.

In [10]:
def is_suspicious_short(term):
    t = str(term).strip()
    return len(t) <= 2

def has_unusual_symbols(term):
    t = str(term)
    # keep common finance/company punctuation relatively permissive
    return bool(re.search(r"[^A-Za-z0-9\s\-/&().,%:+\"]", t))

df_aspect["aspect_empty"] = df_aspect["aspect_term_norm"].str.strip().eq("")
df_aspect["aspect_too_short"] = df_aspect["aspect_term_norm"].apply(is_suspicious_short)
df_aspect["aspect_has_unusual_symbols"] = df_aspect["aspect_term_norm"].apply(has_unusual_symbols)

In [11]:
dup_within_headline = (
    df_aspect.groupby(["raw_id", "aspect_term_norm"])
    .size()
    .reset_index(name="count")
)
dup_within_headline = dup_within_headline[dup_within_headline["count"] > 1]

print("Empty aspect keys:", int(df_aspect["aspect_empty"].sum()))
print("Very short aspect keys:", int(df_aspect["aspect_too_short"].sum()))
print("Aspect keys with unusual symbols:", int(df_aspect["aspect_has_unusual_symbols"].sum()))
print("Duplicate aspect keys within same headline:", len(dup_within_headline))

Empty aspect keys: 0
Very short aspect keys: 80
Aspect keys with unusual symbols: 15
Duplicate aspect keys within same headline: 0


In [12]:
df_aspect.loc[
    df_aspect["aspect_too_short"] | df_aspect["aspect_has_unusual_symbols"] | df_aspect["aspect_empty"],
    ["raw_id", "title_norm", "aspect_term_raw", "aspect_term_norm", "sentiment"]
].head(30)

,raw_id,title_norm,aspect_term_raw,aspect_term_norm,sentiment
21,17,Why Chinese stocks leave US investors vulnerable,US,US,negative
225,202,Put OI concentration at 6000 shows support,OI,OI,neutral
363,321,Stock-specific action likely on IT counters: N...,IT,IT,neutral
364,322,Stock-specific action likely on IT counters: N...,IT,IT,neutral
970,858,GE appoints Munesh Makhija as MD of India tech...,GE,GE,neutral
1167,1027,All is not lost for investors; IT is analysts'...,IT,IT,positive
1401,1227,Current market situations point to a mixed bag...,IT,IT,neutral
1630,1410,IBJA hires consultancy firm EY to draw bluepri...,EY,EY,neutral
1797,1551,"Focus turns to US outlook, Russian stocks stab...",US,US,positive
1799,1552,"Focus turns to US outlook, Russian stocks stab...",US,US,positive


## Step 6. Define pseudo-span matching policy

Because this finance dataset does not provide gold character spans, we generate pseudo-spans by grounding aspect terms back to the headline text.

Matching priority:

1. unique case-insensitive whole-word match
2. unique exact substring match
3. unique normalized match
4. otherwise mark as ambiguous and send to review queue

This phase does not silently force uncertain matches.

In [13]:
def normalize_for_match(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def find_whole_word_matches(text, term):
    # case-insensitive whole-word-ish regex
    pattern = re.compile(rf"(?<!\w){re.escape(term)}(?!\w)", flags=re.IGNORECASE)
    return [(m.start(), m.end()) for m in pattern.finditer(text)]

def find_exact_substring_matches(text, term):
    text_low = text.lower()
    term_low = term.lower()
    matches = []
    start = 0
    while True:
        idx = text_low.find(term_low, start)
        if idx == -1:
            break
        matches.append((idx, idx + len(term)))
        start = idx + 1
    return matches

def find_normalized_match(text, term):
    # conservative fallback:
    # try exact substring after whitespace normalization only
    text_norm = re.sub(r"\s+", " ", text).strip()
    term_norm = re.sub(r"\s+", " ", term).strip()
    matches = find_exact_substring_matches(text_norm, term_norm)
    return matches

def match_aspect_span(text, term):
    text = str(text)
    term = str(term).strip()
    
    if term == "":
        return {
            "match_status": "review",
            "match_type": "empty_term",
            "char_from": None,
            "char_to": None,
            "num_candidates": 0
        }
    
    # Level 1: whole-word match
    ww = find_whole_word_matches(text, term)
    if len(ww) == 1:
        return {
            "match_status": "matched",
            "match_type": "whole_word_unique",
            "char_from": ww[0][0],
            "char_to": ww[0][1],
            "num_candidates": 1
        }
    elif len(ww) > 1:
        return {
            "match_status": "review",
            "match_type": "whole_word_ambiguous",
            "char_from": None,
            "char_to": None,
            "num_candidates": len(ww)
        }
    
    # Level 2: exact substring
    ex = find_exact_substring_matches(text, term)
    if len(ex) == 1:
        return {
            "match_status": "matched",
            "match_type": "substring_unique",
            "char_from": ex[0][0],
            "char_to": ex[0][1],
            "num_candidates": 1
        }
    elif len(ex) > 1:
        return {
            "match_status": "review",
            "match_type": "substring_ambiguous",
            "char_from": None,
            "char_to": None,
            "num_candidates": len(ex)
        }
    
    # Level 3: normalized fallback
    nm = find_normalized_match(text, term)
    if len(nm) == 1:
        return {
            "match_status": "matched",
            "match_type": "normalized_unique",
            "char_from": nm[0][0],
            "char_to": nm[0][1],
            "num_candidates": 1
        }
    elif len(nm) > 1:
        return {
            "match_status": "review",
            "match_type": "normalized_ambiguous",
            "char_from": None,
            "char_to": None,
            "num_candidates": len(nm)
        }
    
    return {
        "match_status": "review",
        "match_type": "not_found",
        "char_from": None,
        "char_to": None,
        "num_candidates": 0
    }

In [14]:
match_results = df_aspect.apply(
    lambda r: match_aspect_span(r["title_norm"], r["aspect_term_norm"]),
    axis=1
)

match_df = pd.DataFrame(list(match_results))
df_aspect = pd.concat([df_aspect.reset_index(drop=True), match_df.reset_index(drop=True)], axis=1)

df_aspect.head()

,raw_id,title_norm,aspect_term_raw,aspect_term_norm,sentiment,num_aspects_in_headline,label_signature,aspect_empty,aspect_too_short,aspect_has_unusual_symbols,match_status,match_type,char_from,char_to,num_candidates
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet,SpiceJet,neutral,1,neutral,False,False,False,matched,whole_word_unique,0.0,8.0,1
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC,MMTC,neutral,1,neutral,False,False,False,matched,whole_word_unique,0.0,4.0,1
2,3,"Mid-cap funds can deliver more, stay put: Experts",Mid-cap funds,Mid-cap funds,positive,1,positive,False,False,False,matched,whole_word_unique,0.0,13.0,1
3,4,Mid caps now turn into market darlings,Mid caps,Mid caps,positive,1,positive,False,False,False,matched,whole_word_unique,0.0,8.0,1
4,5,"Market seeing patience, if not conviction: Pra...",Market,Market,neutral,1,neutral,False,False,False,matched,whole_word_unique,0.0,6.0,1


## Step 7. Validate generated spans

For matched cases, verify that the recovered span text corresponds back to the aspect term.
This is a critical self-check before downstream BIO conversion.

In [15]:
def recover_span_text(row):
    if row["match_status"] != "matched":
        return None
    return row["title_norm"][int(row["char_from"]):int(row["char_to"])]

df_aspect["recovered_span_text"] = df_aspect.apply(recover_span_text, axis=1)

def span_recovery_ok(row):
    if row["match_status"] != "matched":
        return None
    a = normalize_for_match(row["recovered_span_text"])
    b = normalize_for_match(row["aspect_term_norm"])
    return a == b

df_aspect["span_recovery_ok"] = df_aspect.apply(span_recovery_ok, axis=1)

matched_only = df_aspect["match_status"] == "matched"
recovery_ok_ratio = df_aspect.loc[matched_only, "span_recovery_ok"].mean()

print("Matched rows:", int(matched_only.sum()))
print("Recovery OK ratio on matched rows:", recovery_ok_ratio)

Matched rows: 14382
Recovery OK ratio on matched rows: 1.0


In [16]:
df_aspect.loc[
    matched_only & (~df_aspect["span_recovery_ok"].fillna(True)),
    ["raw_id", "title_norm", "aspect_term_norm", "recovered_span_text", "match_type", "char_from", "char_to"]
].head(20)

,raw_id,title_norm,aspect_term_norm,recovered_span_text,match_type,char_from,char_to
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet,SpiceJet,whole_word_unique,0.0,8.0
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC,MMTC,whole_word_unique,0.0,4.0
2,3,"Mid-cap funds can deliver more, stay put: Experts",Mid-cap funds,Mid-cap funds,whole_word_unique,0.0,13.0
3,4,Mid caps now turn into market darlings,Mid caps,Mid caps,whole_word_unique,0.0,8.0
4,5,"Market seeing patience, if not conviction: Pra...",Market,Market,whole_word_unique,0.0,6.0
5,6,Infosys: Will the strong volume growth sustain?,Infosys,Infosys,whole_word_unique,0.0,7.0
6,7,Hudco raises Rs 279 cr via tax-free bonds,Hudco,Hudco,whole_word_unique,0.0,5.0
7,8,HOEC could retest 30-35 levels: Ashwani Gujral,HOEC,HOEC,whole_word_unique,0.0,4.0
8,9,Gold shines on seasonal demand; Silver dull,Gold,Gold,whole_word_unique,0.0,4.0
9,9,Gold shines on seasonal demand; Silver dull,Silver,Silver,whole_word_unique,32.0,38.0


## Step 8. Create a review queue for unresolved or ambiguous matches

Any aspect pair that cannot be matched confidently is exported for manual inspection.
This prevents silent corruption of pseudo-span supervision.

In [17]:
df_review = df_aspect.loc[
    (df_aspect["match_status"] != "matched") |
    (df_aspect["span_recovery_ok"] == False),
    [
        "raw_id", "title_norm", "aspect_term_raw", "aspect_term_norm",
        "sentiment", "match_status", "match_type", "num_candidates"
    ]
].copy()

df_review.to_csv(SPAN_REVIEW_QUEUE_PATH, index=False)
print("Saved review queue to:", SPAN_REVIEW_QUEUE_PATH)
print("Review queue size:", len(df_review))
df_review.head(20)

Saved review queue to: outputs_phase1\span_review_queue.csv
Review queue size: 27


,raw_id,title_norm,aspect_term_raw,aspect_term_norm,sentiment,match_status,match_type,num_candidates
59,50,"Amidst commodity boom, vanilla remains plain v...",vanilla,vanilla,neutral,review,whole_word_ambiguous,2
396,351,BoR awaits legal view for ICICI-BoR merger,BoR,BoR,neutral,review,whole_word_ambiguous,2
2239,1940,BSE to include Arvind Ltd in S&P BSE 200,BSE,BSE,neutral,review,whole_word_ambiguous,2
4384,3744,Gur prices decline on heavy supply of new gur,Gur,Gur,negative,review,whole_word_ambiguous,2
5638,4761,"Add IT, pharma names to portfolio; Mastek, Aur...",pharma,pharma,positive,review,whole_word_ambiguous,2
5907,4862,"EM ASIA FX-Yuan's slide ruffles Asia FX; won, ...",Asia FX,Asia FX,negative,review,whole_word_ambiguous,2
6091,4937,"Dollar supported before jobs data, Aussie doll...",Dollar,Dollar,neutral,review,whole_word_ambiguous,2
6174,4971,EM ASIA FX-Commodity rout weighs on Asia FX; y...,Asia FX,Asia FX,negative,review,whole_word_ambiguous,2
6188,4976,"IT, Pharma under pressure; Infosys, TCS, Sun P...",Pharma,Pharma,negative,review,whole_word_ambiguous,2
7408,5628,SBI to launch FTSE SBI bond indices,SBI,SBI,neutral,review,whole_word_ambiguous,2


## Step 9. Summarize aspect grounding quality

This report measures how close the dataset is to span-based ABSA readiness.

In [18]:
aspect_quality_report = {
    "num_aspect_pairs": int(len(df_aspect)),
    "num_matched": int((df_aspect["match_status"] == "matched").sum()),
    "num_review": int((df_aspect["match_status"] == "review").sum()),
    "match_rate": float((df_aspect["match_status"] == "matched").mean()),
    "match_type_distribution": df_aspect["match_type"].value_counts().to_dict(),
    "recovery_ok_on_matched": float(
        df_aspect.loc[df_aspect["match_status"] == "matched", "span_recovery_ok"].mean()
    ) if (df_aspect["match_status"] == "matched").any() else None
}

pprint(aspect_quality_report)

with open(ASPECT_QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(aspect_quality_report, f, ensure_ascii=False, indent=2)

print("Saved:", ASPECT_QUALITY_PATH)

{'match_rate': 0.9981261711430356,
 'match_type_distribution': {'substring_unique': 89,
                             'whole_word_ambiguous': 27,
                             'whole_word_unique': 14293},
 'num_aspect_pairs': 14409,
 'num_matched': 14382,
 'num_review': 27,
 'recovery_ok_on_matched': 1.0}
Saved: outputs_phase1\aspect_quality_report.json


## Step 10. Prepare duplicate grouping metadata

This step does not split the dataset yet.
It only creates a grouping key so that duplicate or near-duplicate headlines can later be assigned to the same split.

In [19]:
def build_group_key(text):
    text = normalize_for_match(text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["group_key"] = df["title_norm"].apply(build_group_key)

group_sizes = df["group_key"].value_counts()
print("Groups with size > 1:", int((group_sizes > 1).sum()))
group_sizes.head(10)

Groups with size > 1: 68


group_key
several small caps trading at eyepopping valuations        2
pulses prices end steady in tight movements                2
manappuram finance launches another public issue           2
madras stock exchange to close doors                       2
jyothy laboratories rejigs top management                  2
stockspecific action likely on it counters nitin raheja    2
mark mobius expects 1520 returns from indian markets       2
euronext names new head of posttrade business              2
important support for eurusd lies at 1358886               2
dry fruit prices rule steady in tight movements            2
Name: count, dtype: int64

## Step 11. Build the canonical headline-level ABSA dataset

We now aggregate validated aspect-level matches back into a headline-level canonical format.
Only the canonical format should be used by downstream preprocessing.

In [20]:
canonical_records = []

for raw_id, group in df_aspect.groupby("raw_id"):
    headline = group["title_norm"].iloc[0]
    group_key = build_group_key(headline)
    
    aspects = []
    for _, r in group.iterrows():
        aspects.append({
            "term": r["aspect_term_norm"],
            "sentiment": r["sentiment"],
            "char_from": None if pd.isna(r["char_from"]) else int(r["char_from"]),
            "char_to": None if pd.isna(r["char_to"]) else int(r["char_to"]),
            "match_status": r["match_status"],
            "match_type": r["match_type"]
        })
    
    canonical_records.append({
        "raw_id": int(raw_id),
        "doc_id": f"sentfin_{int(raw_id):06d}",
        "text": headline,
        "group_key": group_key,
        "num_aspects": int(group["num_aspects_in_headline"].iloc[0]),
        "label_signature": group["label_signature"].iloc[0],
        "aspects": aspects
    })

df_canonical = pd.DataFrame(canonical_records)
print(df_canonical.shape)
df_canonical.head()

(10753, 7)


,raw_id,doc_id,text,group_key,num_aspects,label_signature,aspects
0,1,sentfin_000001,SpiceJet to issue 6.4 crore warrants to promoters,spicejet to issue 64 crore warrants to promoters,1,neutral,"[{'term': 'SpiceJet', 'sentiment': 'neutral', ..."
1,2,sentfin_000002,MMTC Q2 net loss at Rs 10.4 crore,mmtc q2 net loss at rs 104 crore,1,neutral,"[{'term': 'MMTC', 'sentiment': 'neutral', 'cha..."
2,3,sentfin_000003,"Mid-cap funds can deliver more, stay put: Experts",midcap funds can deliver more stay put experts,1,positive,"[{'term': 'Mid-cap funds', 'sentiment': 'posit..."
3,4,sentfin_000004,Mid caps now turn into market darlings,mid caps now turn into market darlings,1,positive,"[{'term': 'Mid caps', 'sentiment': 'positive',..."
4,5,sentfin_000005,"Market seeing patience, if not conviction: Pra...",market seeing patience if not conviction praka...,1,neutral,"[{'term': 'Market', 'sentiment': 'neutral', 'c..."


In [21]:
# CSV version (useful for quick spreadsheet-like viewing)
df_canonical.to_csv(CANONICAL_CSV_PATH, index=False)
print("Saved canonical CSV:", CANONICAL_CSV_PATH)

# JSONL version (preferred source of truth for next stage)
with open(CANONICAL_JSONL_PATH, "w", encoding="utf-8") as f:
    for rec in canonical_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved canonical JSONL:", CANONICAL_JSONL_PATH)

Saved canonical CSV: outputs_phase1\finance_absa_phase1_canonical.csv
Saved canonical JSONL: outputs_phase1\finance_absa_phase1_canonical.jsonl


## Step 12. Save the Phase 1 report

This report summarizes whether the dataset is now ABSA-ready enough to proceed to controlled splitting and task-specific export.

In [22]:
phase1_report = {
    "input_rows": int(len(df)),
    "aspect_rows": int(len(df_aspect)),
    "words_validation": words_summary,
    "aspect_quality": aspect_quality_report,
    "num_review_queue": int(len(df_review)),
    "num_group_keys": int(df["group_key"].nunique()),
    "canonical_csv_exported": CANONICAL_CSV_PATH.exists(),
    "canonical_jsonl_exported": CANONICAL_JSONL_PATH.exists(),
    "review_queue_exported": SPAN_REVIEW_QUEUE_PATH.exists()
}

with open(PHASE1_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(phase1_report, f, ensure_ascii=False, indent=2)

print("Saved:", PHASE1_REPORT_PATH)
pprint(phase1_report)

Saved: outputs_phase1\phase1_report.json
{'aspect_quality': {'match_rate': 0.9981261711430356,
                    'match_type_distribution': {'substring_unique': 89,
                                                'whole_word_ambiguous': 27,
                                                'whole_word_unique': 14293},
                    'num_aspect_pairs': 14409,
                    'num_matched': 14382,
                    'num_review': 27,
                    'recovery_ok_on_matched': 1.0},
 'aspect_rows': 14409,
 'canonical_csv_exported': True,
 'canonical_jsonl_exported': True,
 'input_rows': 10753,
 'num_group_keys': 10685,
 'num_review_queue': 27,
 'review_queue_exported': True,
 'words_validation': {'exact_match_count': 10665,
                      'exact_match_ratio': 0.9918162373291175,
                      'max_abs_diff': 3.0,
                      'mean_abs_diff': 0.008555751883195387,
                      'rows_with_numeric_words': 10753}}


## Step 13. Evaluate whether Phase 1 is complete

Phase 1 is complete if:
- the audited CSV can be loaded;
- aspect dictionaries can be reconstructed;
- `Words` has been evaluated;
- aspect keys have been audited;
- pseudo-span matching has been attempted systematically;
- ambiguous cases are isolated into a review queue;
- canonical outputs are exported.

In [23]:
checklist = [
    ("Input audited CSV loaded", len(df) > 0),
    ("Aspect dictionaries reconstructed", df["decisions_dict"].apply(lambda x: isinstance(x, dict)).all()),
    ("Words validation report exported", WORDS_REPORT_PATH.exists()),
    ("Aspect quality report exported", ASPECT_QUALITY_PATH.exists()),
    ("Review queue exported", SPAN_REVIEW_QUEUE_PATH.exists()),
    ("Canonical CSV exported", CANONICAL_CSV_PATH.exists()),
    ("Canonical JSONL exported", CANONICAL_JSONL_PATH.exists()),
    ("At least some aspect pairs were matched", (df_aspect["match_status"] == "matched").any()),
]

df_checklist = pd.DataFrame(checklist, columns=["check_item", "passed"])
df_checklist.to_csv(PHASE1_CHECKLIST_PATH, index=False)

display(df_checklist)

all_passed = df_checklist["passed"].all()
print("PHASE 1 COMPLETE:", all_passed)

,check_item,passed
0,Input audited CSV loaded,True
1,Aspect dictionaries reconstructed,True
2,Words validation report exported,True
3,Aspect quality report exported,True
4,Review queue exported,True
5,Canonical CSV exported,True
6,Canonical JSONL exported,True
7,At least some aspect pairs were matched,True


PHASE 1 COMPLETE: True


## Phase 1 conclusion

This phase is considered successful if the finance dataset has been converted into a canonical ABSA-ready representation with:

- reconstructed aspect-sentiment annotations,
- validated metadata,
- aspect-level grounding attempts,
- explicit pseudo-spans for reliable matches,
- a review queue for unresolved cases,
- and canonical exports for the next phase.

The next phase will perform:
1. controlled headline-level shuffle and stratified split,
2. duplicate-aware split assignment,
3. task-specific export for ATE and ASC.